# Tversky Minimal Viable Configuration

Goal: find the smallest Tversky projection that still captures meaningful token similarity.

We use synthetic token data with realistic cluster structure and ask:
- How many features do we need before Tversky similarity becomes meaningful?
- How does Tversky compare to a linear projection at each feature count?
- What is the smallest config that still beats linear?
- Does sharing features across layers hurt quality?

This directly informs how small we can make the Tversky component in the actual model.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

DIM = 768
VOCAB = 8192
N_CLUSTERS = 64
N_TRAIN = 4000
N_TEST = 1000
OUT_DIM = 128  # simulating attention output projection dim

print('Setup complete')

## 1. Synthetic token data with cluster structure

In [ ]:
# Cluster centers — simulates semantic groups (verbs, nouns, function words etc)
cluster_centers = torch.randn(N_CLUSTERS, DIM) * 2.0

# Token embeddings: cluster center + noise
tokens_per_cluster = VOCAB // N_CLUSTERS
token_embeddings = torch.zeros(VOCAB, DIM)
for c in range(N_CLUSTERS):
    start = c * tokens_per_cluster
    end = min(start + tokens_per_cluster, VOCAB)
    n = end - start
    token_embeddings[start:end] = cluster_centers[c] + torch.randn(n, DIM) * 0.5

token_embeddings = F.normalize(token_embeddings, dim=-1)

# Zipf-distributed token frequencies like real language
freq = 1.0 / (torch.arange(1, VOCAB+1).float() ** 0.8)
freq = freq / freq.sum()

# Sample sequences and build (hidden_state, next_token_cluster) pairs
# The task: given a hidden state, predict what cluster the next token belongs to
# This simulates the attention output projection task
train_ids = torch.multinomial(freq.expand(N_TRAIN, -1), 2, replacement=True)
test_ids  = torch.multinomial(freq.expand(N_TEST, -1),  2, replacement=True)

train_x = token_embeddings[train_ids[:, 0]]       # current token hidden state
train_y = train_ids[:, 1] % OUT_DIM               # next token target (mapped to OUT_DIM)
test_x  = token_embeddings[test_ids[:, 0]]
test_y  = test_ids[:, 1] % OUT_DIM

print(f'Train: {train_x.shape} -> {train_y.shape}')
print(f'Test:  {test_x.shape}  -> {test_y.shape}')
print(f'Token embedding norm: {token_embeddings.norm(dim=-1).mean():.4f} (should be 1.0)')

## 2. Projection models

In [ ]:
class LinearProj(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_dim, in_dim) * 0.02)
    def forward(self, x, shared_features=None):
        return x @ self.w.t()
    def param_count(self):
        return self.w.numel()


class TverskyProj(nn.Module):
    def __init__(self, in_dim, out_dim, num_features, shared_features=None):
        super().__init__()
        self.out_dim = out_dim
        self.owns_features = shared_features is None
        if self.owns_features:
            self.features = nn.Parameter(torch.empty(num_features, in_dim).uniform_(-0.02, 0.02))
        else:
            self.register_buffer('features', shared_features)
        self.prototypes = nn.Parameter(torch.empty(out_dim, in_dim).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))

    def forward(self, x, shared_features=None):
        feat  = shared_features if shared_features is not None else self.features
        proto = self.prototypes
        x_f = x     @ feat.t()
        p_f = proto  @ feat.t()
        x_s = torch.sigmoid(x_f * 5.0)
        p_s = torch.sigmoid(p_f * 5.0)
        x_a = x_f * x_s
        p_a = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())

    def param_count(self):
        base = self.prototypes.numel() + 3
        if self.owns_features:
            base += self.features.numel()
        return base


def train_eval(model, train_x, train_y, test_x, test_y,
               n_epochs=50, lr=1e-3, batch=256, extra_params=None):
    params = list(model.parameters())
    if extra_params:
        params += extra_params
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(n_epochs):
        idx = torch.randperm(len(train_x))
        for i in range(0, len(train_x), batch):
            b = idx[i:i+batch]
            opt.zero_grad()
            F.cross_entropy(model(train_x[b]), train_y[b]).backward()
            opt.step()
    with torch.no_grad():
        loss = F.cross_entropy(model(test_x), test_y).item()
        acc  = (model(test_x).argmax(-1) == test_y).float().mean().item()
    return loss, acc

print('Models defined')

## 3. Sweep: feature count vs quality

In [ ]:
# Linear baseline
lin = LinearProj(DIM, OUT_DIM)
lin_loss, lin_acc = train_eval(lin, train_x, train_y, test_x, test_y, n_epochs=50)
print(f'Linear: loss={lin_loss:.4f}, acc={lin_acc:.4f}, params={lin.param_count():,}')
print()

feature_sweep = [4, 8, 16, 32, 64, 128, 256, 512]
results = []

for nf in feature_sweep:
    tv = TverskyProj(DIM, OUT_DIM, nf)
    loss, acc = train_eval(tv, train_x, train_y, test_x, test_y, n_epochs=50)
    results.append({'nf': nf, 'loss': loss, 'acc': acc, 'params': tv.param_count()})
    delta = loss - lin_loss
    sign = '✓ BEATS' if loss < lin_loss else '✗ worse'
    print(f'{sign} | {nf:4d} features: loss={loss:.4f} ({delta:+.4f}), acc={acc:.4f}, params={tv.param_count():,}')

In [ ]:
import time

def benchmark(model, x, n_runs=100):
    # warmup
    for _ in range(10):
        _ = model(x)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        _ = model(x)
    return (time.perf_counter() - t0) / n_runs * 1000  # ms

lin_time = benchmark(lin, test_x)
for r in results:
    # rebuild model to benchmark
    tv = TverskyProj(DIM, OUT_DIM, r['nf'])
    tv_time = benchmark(tv, test_x)
    print(f"{r['nf']:4d} features: {tv_time:.3f}ms vs linear {lin_time:.3f}ms ({tv_time/lin_time:.1f}x)")

In [ ]:
class AsymmetricLinearV1(nn.Module):
    """Asymmetric via presence/absence split — 2 relu ops, same params as linear + 1 scalar."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_dim, in_dim) * 0.02)
        self.alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, shared_features=None):
        out = x @ self.w.t()
        out_pos = torch.relu(out)
        out_neg = torch.relu(-out)
        return out_pos - self.alpha.abs() * out_neg
    def param_count(self):
        return self.w.numel() + 1

class AsymmetricLinearV2(nn.Module):
    """Asymmetric via signed gate — 1 matmul + sign op, same params as linear + 1 scalar."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_dim, in_dim) * 0.02)
        self.alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, shared_features=None):
        out = x @ self.w.t()
        gate = torch.sign(out)
        return out * (1 + self.alpha.abs() * gate)
    def param_count(self):
        return self.w.numel() + 1
    
class TverskyHardThresh(TverskyProj):
    def forward(self, x, shared_features=None):
        feat = self.features
        proto = self.prototypes
        x_f = x @ feat.t()
        p_f = proto @ feat.t()
        x_s = (x_f > 0).float()
        p_s = (p_f > 0).float()
        x_a = x_f * x_s
        p_a = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())

class TverskyPolyApprox(TverskyProj):
    def forward(self, x, shared_features=None):
        feat = self.features
        proto = self.prototypes
        x_f = x @ feat.t()
        p_f = proto @ feat.t()
        x_s = torch.clamp(x_f * 5.0 / 4 + 0.5, 0, 1)
        p_s = torch.clamp(p_f * 5.0 / 4 + 0.5, 0, 1)
        x_a = x_f * x_s
        p_a = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())

# Train and benchmark all three
models = {
    'Linear': LinearProj(DIM, OUT_DIM),
    'AsymLinearV1 (relu split)': AsymmetricLinearV1(DIM, OUT_DIM),
    'AsymLinearV2 (sign gate)': AsymmetricLinearV2(DIM, OUT_DIM),
    'Tversky 4f': TverskyProj(DIM, OUT_DIM, 4),
    'Tversky 8f': TverskyProj(DIM, OUT_DIM, 8),
    'TverskyHard 4f': TverskyHardThresh(DIM, OUT_DIM, 4),
    'TverskyPoly 4f': TverskyPolyApprox(DIM, OUT_DIM, 4),
}

print(f'{"Model":<30} {"Loss":>8} {"vs Linear":>10} {"Time ms":>9} {"vs Linear":>10} {"Params":>10}')
print('-' * 80)

lin_ref = LinearProj(DIM, OUT_DIM)
lin_loss_ref, _ = train_eval(lin_ref, train_x, train_y, test_x, test_y, n_epochs=50)
lin_time_ref = benchmark(lin_ref, test_x)

for name, model in models.items():
    loss, acc = train_eval(model, train_x, train_y, test_x, test_y, n_epochs=50)
    t = benchmark(model, test_x)
    print(f'{name:<30} {loss:>8.4f} {loss-lin_loss_ref:>+10.4f} {t:>9.3f} {t/lin_time_ref:>9.1f}x {model.param_count():>10,}')

## 4. Shared features: how much does sharing hurt?

In [ ]:
# With shared features, the feature bank is learned jointly across N_LAYERS projections
# Here we simulate this with 1 projection + shared feature bank
# (in the real model, multiple layers would share the same bank)

N_LAYERS = 12

print(f'Shared vs per-layer features ({N_LAYERS} layers budget context):')
print()

for nf in [16, 32, 64, 128]:
    # Per-layer (independent features per projection)
    tv_own = TverskyProj(DIM, OUT_DIM, nf)
    loss_own, _ = train_eval(tv_own, train_x, train_y, test_x, test_y, n_epochs=50)

    # Shared features (one bank, trained alongside the layer)
    shared_feat = nn.Parameter(torch.empty(nf, DIM).uniform_(-0.02, 0.02))
    tv_shared = TverskyProj(DIM, OUT_DIM, nf)
    
    # Train with shared features passed explicitly
    opt = torch.optim.Adam(list(tv_shared.parameters()) + [shared_feat], lr=1e-3)
    for _ in range(50):
        idx = torch.randperm(len(train_x))
        for i in range(0, len(train_x), 256):
            b = idx[i:i+256]
            opt.zero_grad()
            F.cross_entropy(tv_shared(train_x[b], shared_feat), train_y[b]).backward()
            opt.step()
    with torch.no_grad():
        loss_shared = F.cross_entropy(tv_shared(test_x, shared_feat), test_y).item()

    # Budget for N_LAYERS
    cost_own_mb    = N_LAYERS * (nf*DIM + DIM*OUT_DIM) * 2 / 1e6
    cost_shared_mb = (nf*DIM + N_LAYERS * DIM*OUT_DIM) * 2 / 1e6
    saving_mb      = cost_own_mb - cost_shared_mb

    print(f'{nf:4d} features:')
    print(f'  per-layer loss={loss_own:.4f} | shared loss={loss_shared:.4f} | delta={loss_shared-loss_own:+.4f}')
    print(f'  budget: per-layer={cost_own_mb:.2f}MB | shared={cost_shared_mb:.2f}MB | saving={saving_mb:.2f}MB')
    print()

## 5. Summary: minimum viable Tversky

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

nfs   = [r['nf']   for r in results]
losses = [r['loss'] for r in results]
params = [r['params'] for r in results]

axes[0].plot(nfs, losses, 'bo-', label='Tversky')
axes[0].axhline(lin_loss, color='r', linestyle='--', label=f'Linear ({lin_loss:.4f})')
axes[0].set_xlabel('Number of features')
axes[0].set_ylabel('Test loss')
axes[0].set_title('Quality vs feature count')
axes[0].set_xscale('log', base=2)
axes[0].legend()

axes[1].scatter(params, losses, c='blue', label='Tversky')
axes[1].axhline(lin_loss, color='r', linestyle='--', label='Linear')
axes[1].scatter([lin.param_count()], [lin_loss], c='red', marker='*', s=200)
for r in results:
    axes[1].annotate(str(r['nf']), (r['params'], r['loss']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[1].set_xlabel('Param count')
axes[1].set_ylabel('Test loss')
axes[1].set_title('Params vs quality')
axes[1].legend()

plt.tight_layout()
plt.savefig('tversky_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== SUMMARY ===')
print(f'Linear baseline: loss={lin_loss:.4f}')
for r in results:
    sign = 'BEATS' if r['loss'] < lin_loss else 'loses'
    print(f'{r["nf"]:4d} features: {sign} linear by {lin_loss - r["loss"]:+.4f}')

beats = [r for r in results if r['loss'] < lin_loss]
if beats:
    best = min(beats, key=lambda r: r['nf'])
    print(f'\nMinimum features to beat linear: {best["nf"]}')
    extra_params_vs_linear = best['params'] - lin.param_count()
    print(f'Extra params vs linear: {extra_params_vs_linear:,} ({extra_params_vs_linear*2/1e6:.3f}MB fp16)')
else:
    print('\nTversky did not beat linear — may need more training or data complexity')

In [ ]:
# ============================================================
# Directional Relationship Task
# Tests whether Tversky captures asymmetric similarity better
# than linear when direction genuinely matters
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

# --- Config ---
DIM = 768
N_CONCEPTS = 300
N_TRAIN = 3000
N_TEST = 1000
N_EPOCHS = 80
LR = 1e-3
BATCH = 256

# --- Build hierarchical concepts ---
# Each concept has features from ancestors + unique ones
# Directionality: parent has FEWER features than child (child specialises parent)
concepts = torch.zeros(N_CONCEPTS, DIM)
n_roots = N_CONCEPTS // 3

for i in range(N_CONCEPTS):
    level = i % 3
    if level == 0:
        active = torch.randperm(DIM)[:16]
    elif level == 1:
        parent = (i // 3) * 3
        parent_active = concepts[parent].nonzero().squeeze(-1)
        new = torch.randperm(DIM)[:12]
        active = torch.cat([parent_active, new])[:28]
    else:
        parent = ((i // 3) * 3) + 1
        parent_active = concepts[parent].nonzero().squeeze(-1)
        new = torch.randperm(DIM)[:10]
        active = torch.cat([parent_active, new])[:40]
    active = active.clamp(0, DIM - 1).unique()
    concepts[i, active] = torch.randn(len(active)).abs() + 0.5

concepts = F.normalize(concepts, dim=-1)

# --- Generate directional pairs ---
# Label 0: A is parent of B (A more general, B specialises A)
# Label 1: B is parent of A (B more general, A specialises B)
# Label 2: unrelated
def make_pairs(n):
    xs, ys, labels = [], [], []
    for _ in range(n):
        r = torch.randint(0, 3, (1,)).item()
        if r == 0:  # A=parent, B=child
            parent_idx = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            child_idx = parent_idx + torch.randint(1, 3, (1,)).item()
            xs.append(concepts[parent_idx])
            ys.append(concepts[child_idx])
            labels.append(0)
        elif r == 1:  # B=parent, A=child (reversed)
            parent_idx = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            child_idx = parent_idx + torch.randint(1, 3, (1,)).item()
            xs.append(concepts[child_idx])
            ys.append(concepts[parent_idx])
            labels.append(1)
        else:  # unrelated
            i, j = torch.randint(0, N_CONCEPTS, (2,)).tolist()
            xs.append(concepts[i])
            ys.append(concepts[j])
            labels.append(2)
    return torch.stack(xs), torch.stack(ys), torch.tensor(labels)

train_x, train_y_ctx, train_labels = make_pairs(N_TRAIN)
test_x,  test_y_ctx,  test_labels  = make_pairs(N_TEST)

# Concatenate pair as input: [a, b] -> 2*DIM input
train_in = torch.cat([train_x, train_y_ctx], dim=-1)
test_in  = torch.cat([test_x,  test_y_ctx],  dim=-1)

print(f'Task: 3-way directional classification (A>B, B>A, unrelated)')
print(f'Input dim: {train_in.shape[1]}, Classes: 3')
print(f'Label distribution: {[(test_labels==i).sum().item() for i in range(3)]}')

# --- Models ---
IN = DIM * 2
OUT = 3

class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.randn(OUT, IN) * 0.02)
    def forward(self, x):
        return x @ self.w.t()
    def param_count(self):
        return self.w.numel()

class TverskyClassifier(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.features  = nn.Parameter(torch.empty(num_features, IN).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        feat  = self.features
        proto = self.prototypes
        x_f = x     @ feat.t()
        p_f = proto @ feat.t()
        x_s = torch.sigmoid(x_f * 5.0)
        p_s = torch.sigmoid(p_f * 5.0)
        x_a = x_f * x_s
        p_a = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self):
        return self.features.numel() + self.prototypes.numel() + 3

class TverskyPolyClassifier(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.features   = nn.Parameter(torch.empty(num_features, IN).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        feat  = self.features
        proto = self.prototypes
        x_f = x     @ feat.t()
        p_f = proto @ feat.t()
        x_s = torch.clamp(x_f * 5.0 / 4 + 0.5, 0, 1)
        p_s = torch.clamp(p_f * 5.0 / 4 + 0.5, 0, 1)
        x_a = x_f * x_s
        p_a = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self):
        return self.features.numel() + self.prototypes.numel() + 3

def train_eval_dir(model, n_epochs=N_EPOCHS):
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for _ in range(n_epochs):
        idx = torch.randperm(N_TRAIN)
        for i in range(0, N_TRAIN, BATCH):
            b = idx[i:i+BATCH]
            opt.zero_grad()
            F.cross_entropy(model(train_in[b]), train_labels[b]).backward()
            opt.step()
    with torch.no_grad():
        logits = model(test_in)
        loss = F.cross_entropy(logits, test_labels).item()
        acc  = (logits.argmax(-1) == test_labels).float().mean().item()
        # Per-class accuracy to see if directionality is captured
        acc0 = (logits[test_labels==0].argmax(-1) == 0).float().mean().item()
        acc1 = (logits[test_labels==1].argmax(-1) == 1).float().mean().item()
        acc2 = (logits[test_labels==2].argmax(-1) == 2).float().mean().item()
    return loss, acc, acc0, acc1, acc2

def benchmark_dir(model, n_runs=200):
    for _ in range(20): model(test_in)
    t0 = time.perf_counter()
    for _ in range(n_runs): model(test_in)
    return (time.perf_counter() - t0) / n_runs * 1000

models = {
    'Linear':           LinearClassifier(),
    'Tversky 4f':       TverskyClassifier(4),
    'Tversky 8f':       TverskyClassifier(8),
    'TverskyPoly 4f':   TverskyPolyClassifier(4),
    'TverskyPoly 8f':   TverskyPolyClassifier(8),
}

print(f'\n{"Model":<22} {"Loss":>6} {"Acc":>6} {"A>B":>6} {"B>A":>6} {"Unrel":>6} {"ms":>7} {"Params":>10}')
print('-' * 80)

lin_time = None
for name, model in models.items():
    loss, acc, a0, a1, a2 = train_eval_dir(model)
    t = benchmark_dir(model)
    if lin_time is None:
        lin_time = t
    print(f'{name:<22} {loss:>6.3f} {acc:>6.3f} {a0:>6.3f} {a1:>6.3f} {a2:>6.3f} {t:>6.2f}ms {model.param_count():>10,}')

In [ ]:
# ============================================================
# Optimised Tversky variants — speed + quality + storage
# Goal: beat linear on directional tasks, faster than sigmoid Tversky
# Larger dataset, multiple independent layers, storage analysis
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

DIM = 768
N_CONCEPTS = 500
N_TRAIN = 10000
N_TEST = 2000
N_EPOCHS = 100
LR = 1e-3
BATCH = 512
N_LAYERS = 12  # simulate shared features across 12 layers
OUT = 3
IN = DIM * 2

# --- Rebuild larger hierarchical dataset ---
concepts = torch.zeros(N_CONCEPTS, DIM)
for i in range(N_CONCEPTS):
    level = i % 3
    if level == 0:
        active = torch.randperm(DIM)[:16]
    elif level == 1:
        parent = (i // 3) * 3
        parent_active = concepts[parent].nonzero().squeeze(-1)
        new = torch.randperm(DIM)[:12]
        active = torch.cat([parent_active, new])[:28]
    else:
        parent = ((i // 3) * 3) + 1
        if parent < N_CONCEPTS:
            parent_active = concepts[parent].nonzero().squeeze(-1)
        else:
            parent_active = torch.tensor([])
        new = torch.randperm(DIM)[:10]
        active = torch.cat([parent_active.flatten(), new])[:40]
    active = active.long().clamp(0, DIM-1).unique()
    concepts[i, active] = torch.randn(len(active)).abs() + 0.5
concepts = F.normalize(concepts, dim=-1)

def make_pairs(n):
    xs, ys, labels = [], [], []
    for _ in range(n):
        r = torch.randint(0, 3, (1,)).item()
        if r == 0:
            p = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            c = p + torch.randint(1, 3, (1,)).item()
            c = min(c, N_CONCEPTS - 1)
            xs.append(concepts[p]); ys.append(concepts[c]); labels.append(0)
        elif r == 1:
            p = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            c = p + torch.randint(1, 3, (1,)).item()
            c = min(c, N_CONCEPTS - 1)
            xs.append(concepts[c]); ys.append(concepts[p]); labels.append(1)
        else:
            i, j = torch.randint(0, N_CONCEPTS, (2,)).tolist()
            xs.append(concepts[i]); ys.append(concepts[j]); labels.append(2)
    return torch.stack(xs), torch.stack(ys), torch.tensor(labels)

train_x, train_y_ctx, train_labels = make_pairs(N_TRAIN)
test_x,  test_y_ctx,  test_labels  = make_pairs(N_TEST)
train_in = torch.cat([train_x, train_y_ctx], dim=-1)
test_in  = torch.cat([test_x,  test_y_ctx],  dim=-1)

def storage_mb(num_features, n_layers=N_LAYERS, shared=True):
    """fp16 storage cost for features + prototypes across n_layers."""
    feat_cost  = num_features * IN * 2  # shared once if shared=True
    proto_cost = OUT * IN * 2 * n_layers
    if shared:
        return (feat_cost + proto_cost) / 1e6
    else:
        return (feat_cost * n_layers + proto_cost) / 1e6

def storage_linear(n_layers=N_LAYERS):
    return OUT * IN * 2 * n_layers / 1e6

# --- Model definitions ---

class LinearCls(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.randn(OUT, IN) * 0.02)
    def forward(self, x): return x @ self.w.t()
    def param_count(self): return self.w.numel()

class TverskySigmoid(nn.Module):
    """Original Tversky with sigmoid membership."""
    def __init__(self, nf):
        super().__init__()
        self.features   = nn.Parameter(torch.empty(nf, IN).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
        self.nf = nf
    def forward(self, x):
        x_f = x @ self.features.t()
        p_f = self.prototypes @ self.features.t()
        x_s = torch.sigmoid(x_f * 5.0)
        p_s = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.features.numel() + self.prototypes.numel() + 3

class TverskyReLU(nn.Module):
    """Replace sigmoid with relu normalised by max — cheap, differentiable, no exp."""
    def __init__(self, nf):
        super().__init__()
        self.features   = nn.Parameter(torch.empty(nf, IN).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
        self.nf = nf
    def _membership(self, t):
        pos = torch.relu(t)
        return pos / (pos.amax(dim=-1, keepdim=True).clamp(min=1e-8))
    def forward(self, x):
        x_f = x @ self.features.t()
        p_f = self.prototypes @ self.features.t()
        x_s = self._membership(x_f)
        p_s = self._membership(p_f)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.features.numel() + self.prototypes.numel() + 3

class TverskyHardSTE(nn.Module):
    """Hard threshold membership with STE — binary set membership, fastest possible."""
    def __init__(self, nf):
        super().__init__()
        self.features   = nn.Parameter(torch.empty(nf, IN).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
        self.nf = nf
    def _membership(self, t):
        # Forward: hard threshold. Backward: straight-through
        hard = (t > 0).float()
        return hard + (t.sigmoid() - t.sigmoid().detach())
    def forward(self, x):
        x_f = x @ self.features.t()
        p_f = self.prototypes @ self.features.t()
        x_s = self._membership(x_f)
        p_s = self._membership(p_f)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.features.numel() + self.prototypes.numel() + 3

class TverskyNoFeatures(nn.Module):
    """Tversky WITHOUT a separate feature bank.
    Prototypes serve as their own feature space via cross-similarity.
    Same storage as linear + 3 scalars."""
    def __init__(self):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f   = x @ self.prototypes.t()             # [BATCH, OUT]
        p_norm = F.normalize(self.prototypes, dim=-1)
        p_f   = p_norm @ p_norm.t()                 # [OUT, OUT]
        x_s   = torch.sigmoid(x_f * 5.0)            # [BATCH, OUT]
        p_s   = torch.sigmoid(p_f * 5.0)            # [OUT, OUT]
        x_a   = x_f * x_s                           # [BATCH, OUT]
        p_a   = p_f * p_s                           # [OUT, OUT]
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3

class TverskySharedFeatures(nn.Module):
    """Tversky with features passed in — for shared pool across layers."""
    def __init__(self, nf, shared_feat):
        super().__init__()
        self.shared_feat = shared_feat  # not owned, not counted in params
        self.prototypes  = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
        self.nf = nf
    def forward(self, x):
        feat = self.shared_feat
        x_f = x @ feat.t()
        p_f = self.prototypes @ feat.t()
        x_s = torch.sigmoid(x_f * 5.0)
        p_s = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3  # features not counted (shared)

# --- Training ---
def train_eval_model(model, extra_params=None, n_epochs=N_EPOCHS):
    params = list(model.parameters())
    if extra_params: params += extra_params
    opt = torch.optim.Adam(params, lr=LR)
    for _ in range(n_epochs):
        idx = torch.randperm(N_TRAIN)
        for i in range(0, N_TRAIN, BATCH):
            b = idx[i:i+BATCH]
            opt.zero_grad()
            F.cross_entropy(model(train_in[b]), train_labels[b]).backward()
            opt.step()
    with torch.no_grad():
        logits = model(test_in)
        loss = F.cross_entropy(logits, test_labels).item()
        acc  = (logits.argmax(-1) == test_labels).float().mean().item()
        acc0 = (logits[test_labels==0].argmax(-1) == 0).float().mean().item()
        acc1 = (logits[test_labels==1].argmax(-1) == 1).float().mean().item()
        acc2 = (logits[test_labels==2].argmax(-1) == 2).float().mean().item()
    return loss, acc, acc0, acc1, acc2

def benchmark_model(model, n_runs=500):
    for _ in range(50): model(test_in)
    t0 = time.perf_counter()
    for _ in range(n_runs): model(test_in)
    return (time.perf_counter() - t0) / n_runs * 1000

# --- Shared feature pool ---
shared_feat_4  = nn.Parameter(torch.empty(4,  IN).uniform_(-0.02, 0.02))
shared_feat_8  = nn.Parameter(torch.empty(8,  IN).uniform_(-0.02, 0.02))
shared_feat_16 = nn.Parameter(torch.empty(16, IN).uniform_(-0.02, 0.02))

shared_model_4  = TverskySharedFeatures(4,  shared_feat_4)
shared_model_8  = TverskySharedFeatures(8,  shared_feat_8)
shared_model_16 = TverskySharedFeatures(16, shared_feat_16)

models = {
    'Linear':                  (LinearCls(),          None),
    'Tversky sigmoid 4f':      (TverskySigmoid(4),    None),
    'Tversky sigmoid 8f':      (TverskySigmoid(8),    None),
    'Tversky relu 4f':         (TverskyReLU(4),       None),
    'Tversky relu 8f':         (TverskyReLU(8),       None),
    'Tversky hardSTE 4f':      (TverskyHardSTE(4),    None),
    'Tversky noFeatures':      (TverskyNoFeatures(),  None),
    'Tversky shared 4f':       (shared_model_4,       [shared_feat_4]),
    'Tversky shared 8f':       (shared_model_8,       [shared_feat_8]),
    'Tversky shared 16f':      (shared_model_16,      [shared_feat_16]),
}

print(f'\n{"Model":<26} {"Loss":>6} {"Acc":>6} {"A>B":>6} {"B>A":>6} {"Unr":>6} {"ms":>6} {"StorMB(12L)":>12}')
print('-' * 90)

lin_time = None
lin_storage = storage_linear()

for name, (model, extra) in models.items():
    loss, acc, a0, a1, a2 = train_eval_model(model, extra)
    t = benchmark_model(model)
    if lin_time is None: lin_time = t

    # Storage calculation
    if 'noFeatures' in name:
        stor = OUT * IN * 2 * N_LAYERS / 1e6
    elif 'shared' in name:
        nf = int(name.split(' ')[-1].replace('f',''))
        stor = storage_mb(nf, shared=True)
    elif hasattr(model, 'nf'):
        stor = storage_mb(model.nf, shared=False)
    else:
        stor = lin_storage

    print(f'{name:<26} {loss:>6.3f} {acc:>6.3f} {a0:>6.3f} {a1:>6.3f} {a2:>6.3f} {t:>5.2f}ms {stor:>10.3f}MB')

print(f'\nLinear storage ({N_LAYERS}L): {lin_storage:.3f}MB')
print(f'Note: shared storage splits feature cost once across all {N_LAYERS} layers')

In [ ]:
# Shape verification for clean TverskyNoFeatures
OUT, IN, BATCH = 3, 1536, 32
proto = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
x = torch.randn(BATCH, IN)

x_f   = x @ proto.t()                          # [BATCH, OUT]
p_norm = F.normalize(proto, dim=-1)
p_f   = p_norm @ p_norm.t()                    # [OUT, OUT]

x_s   = torch.sigmoid(x_f * 5.0)               # [BATCH, OUT]
p_s   = torch.sigmoid(p_f * 5.0)               # [OUT, OUT]
x_a   = x_f * x_s                              # [BATCH, OUT]
p_a   = p_f * p_s                              # [OUT, OUT]

# Three terms
term1 = x_a @ p_a.t()                          # [BATCH, OUT] @ [OUT, OUT] -> [BATCH, OUT]
term2 = x_a @ (1 - p_s).t()                    # [BATCH, OUT] @ [OUT, OUT] -> [BATCH, OUT]
term3 = (1 - x_s) @ p_a.t()                    # [BATCH, OUT] @ [OUT, OUT] -> [BATCH, OUT] -- wait, p_a is [OUT,OUT]

print(f'x_f: {x_f.shape}')
print(f'p_f: {p_f.shape}')
print(f'term1: {term1.shape}')
print(f'term2: {term2.shape}')
print(f'term3: {term3.shape}')
print('All shapes correct' if term1.shape == (BATCH, OUT) else 'SHAPE ERROR')

In [ ]:
# ============================================================
# Optimised Tversky variants — Round 2
# Focus: close the speed gap on TverskyNoFeatures,
#        push shared features further, add avg accuracy
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

DIM = 768
N_TRAIN = 50000
N_TEST = 5000
N_EPOCHS = 100
LR = 1e-3
BATCH = 512
N_LAYERS = 12
OUT = 3
IN = DIM * 2

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# More complex hierarchy: 4 levels, more features, partial overlaps
N_CONCEPTS = 800
concepts = torch.zeros(N_CONCEPTS, DIM)
feature_noise = 0.3  # add noise to features so membership is graded not binary

for i in range(N_CONCEPTS):
    level = i % 4
    if level == 0:   # root: 8 features
        active = torch.randperm(DIM)[:8]
        vals = torch.randn(len(active)).abs() + 1.0
    elif level == 1:  # inherits 6 of parent's 8 + 12 new
        parent = (i // 4) * 4
        parent_active = concepts[parent].nonzero().squeeze(-1)
        inherit = parent_active[torch.randperm(len(parent_active))[:6]]
        new = torch.randperm(DIM)[:12]
        active = torch.cat([inherit, new]).unique()
        vals = torch.randn(len(active)).abs() + 0.8
    elif level == 2:  # inherits 10 of parent's + 16 new
        parent = (i // 4) * 4 + 1
        parent = min(parent, N_CONCEPTS - 1)
        parent_active = concepts[parent].nonzero().squeeze(-1)
        inherit = parent_active[torch.randperm(len(parent_active))[:min(10, len(parent_active))]]
        new = torch.randperm(DIM)[:16]
        active = torch.cat([inherit, new]).unique()
        vals = torch.randn(len(active)).abs() + 0.6
    else:             # leaf: inherits 14 of parent's + 20 new + noise
        parent = (i // 4) * 4 + 2
        parent = min(parent, N_CONCEPTS - 1)
        parent_active = concepts[parent].nonzero().squeeze(-1)
        inherit = parent_active[torch.randperm(len(parent_active))[:min(14, len(parent_active))]]
        new = torch.randperm(DIM)[:20]
        active = torch.cat([inherit, new]).unique()
        vals = torch.randn(len(active)).abs() + 0.4
    active = active.long().clamp(0, DIM - 1)
    concepts[i, active] = vals[:len(active)]
    # Add graded noise to all dimensions
    concepts[i] += torch.randn(DIM) * feature_noise

concepts = F.normalize(concepts, dim=-1)
# After building concepts:
concepts = concepts.to(device)


def make_pairs(n):
    xs, ys, labels = [], [], []
    for _ in range(n):
        r = torch.randint(0, 3, (1,)).item()
        if r == 0:
            p = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            c = min(p + torch.randint(1, 3, (1,)).item(), N_CONCEPTS - 1)
            xs.append(concepts[p]); ys.append(concepts[c]); labels.append(0)
        elif r == 1:
            p = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            c = min(p + torch.randint(1, 3, (1,)).item(), N_CONCEPTS - 1)
            xs.append(concepts[c]); ys.append(concepts[p]); labels.append(1)
        else:
            i, j = torch.randint(0, N_CONCEPTS, (2,)).tolist()
            xs.append(concepts[i]); ys.append(concepts[j]); labels.append(2)
    return torch.stack(xs), torch.stack(ys), torch.tensor(labels)

train_x, train_y_ctx, train_labels = make_pairs(N_TRAIN)
test_x,  test_y_ctx,  test_labels  = make_pairs(N_TEST)
train_in = torch.cat([train_x, train_y_ctx], dim=-1)
test_in  = torch.cat([test_x,  test_y_ctx],  dim=-1)



def storage_mb(feat_params, proto_params, n_layers=N_LAYERS, shared_feat=True):
    feat_cost  = feat_params * 2 * (1 if shared_feat else n_layers)
    proto_cost = proto_params * 2 * n_layers
    return (feat_cost + proto_cost) / 1e6

def storage_linear():
    return OUT * IN * 2 * N_LAYERS / 1e6

# --- Baselines ---
class LinearCls(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.randn(OUT, IN) * 0.02)
    def forward(self, x): return x @ self.w.t()
    def param_count(self): return self.w.numel()

class TverskySigmoid(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.nf = nf
        self.features   = nn.Parameter(torch.empty(nf, IN).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f = x @ self.features.t()
        p_f = self.prototypes @ self.features.t()
        x_s = torch.sigmoid(x_f * 5.0)
        p_s = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.features.numel() + self.prototypes.numel() + 3

class TverskyNoFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f    = x @ self.prototypes.t()
        p_norm = F.normalize(self.prototypes, dim=-1)
        p_f    = p_norm @ p_norm.t()
        x_s    = torch.sigmoid(x_f * 5.0)
        p_s    = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3

# --- New variants ---

class TverskyNoFeatPoly(nn.Module):
    """TverskyNoFeatures with poly sigmoid approx — close speed gap vs linear."""
    def __init__(self):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f    = x @ self.prototypes.t()
        p_norm = F.normalize(self.prototypes, dim=-1)
        p_f    = p_norm @ p_norm.t()
        x_s    = torch.clamp(x_f * 5.0 / 4 + 0.5, 0, 1)
        p_s    = torch.clamp(p_f * 5.0 / 4 + 0.5, 0, 1)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3

class TverskyNoFeatLearnedScale(nn.Module):
    """TverskyNoFeatures with learned sigmoid steepness instead of hardcoded 5.0."""
    def __init__(self):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.scale = nn.Parameter(torch.tensor(5.0))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f    = x @ self.prototypes.t()
        p_norm = F.normalize(self.prototypes, dim=-1)
        p_f    = p_norm @ p_norm.t()
        s      = self.scale.abs().clamp(min=0.1)
        x_s    = torch.sigmoid(x_f * s)
        p_s    = torch.sigmoid(p_f * s)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 4

class TverskyLowRank(nn.Module):
    """Low-rank prototype factorization: W = U @ V, r << IN.
    Saves storage + faster matmul at large IN."""
    def __init__(self, rank=32):
        super().__init__()
        self.rank = rank
        self.U = nn.Parameter(torch.randn(OUT, rank) * 0.02)
        self.V = nn.Parameter(torch.randn(rank, IN) * 0.02)
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        W      = self.U @ self.V                    # [OUT, IN] — materialized once
        x_f    = x @ W.t()
        p_norm = F.normalize(W, dim=-1)
        p_f    = p_norm @ p_norm.t()
        x_s    = torch.sigmoid(x_f * 5.0)
        p_s    = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.U.numel() + self.V.numel() + 3

class TverskySharedCls(nn.Module):
    def __init__(self, nf, shared_feat):
        super().__init__()
        self.shared_feat = shared_feat
        self.prototypes  = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
        self.nf = nf
    def forward(self, x):
        feat = self.shared_feat
        x_f  = x @ feat.t()
        p_f  = self.prototypes @ feat.t()
        x_s  = torch.sigmoid(x_f * 5.0)
        p_s  = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3

# --- Training + eval ---
def train_eval_model(model, extra_params=None, n_epochs=N_EPOCHS):
    params = list(model.parameters())
    if extra_params:
        params += [p for p in extra_params if not any(p is q for q in params)]
    opt = torch.optim.Adam(params, lr=LR)
    for _ in range(n_epochs):
        idx = torch.randperm(N_TRAIN)
        for i in range(0, N_TRAIN, BATCH):
            b = idx[i:i+BATCH]
            opt.zero_grad()
            F.cross_entropy(model(train_in[b]), train_labels[b]).backward()
            opt.step()
    with torch.no_grad():
        logits = model(test_in)
        loss   = F.cross_entropy(logits, test_labels).item()
        preds  = logits.argmax(-1)
        acc    = (preds == test_labels).float().mean().item()
        acc0   = (preds[test_labels==0] == 0).float().mean().item()
        acc1   = (preds[test_labels==1] == 1).float().mean().item()
        acc2   = (preds[test_labels==2] == 2).float().mean().item()
        avg    = (acc0 + acc1 + acc2) / 3  # balanced avg across classes
    return loss, acc, acc0, acc1, acc2, avg

def benchmark_model(model, n_runs=500):
    for _ in range(50): model(test_in)
    t0 = time.perf_counter()
    for _ in range(n_runs): model(test_in)
    return (time.perf_counter() - t0) / n_runs * 1000

# --- Shared feature banks ---
shared_4f  = nn.Parameter(torch.empty(4,  IN).uniform_(-0.02, 0.02))
shared_8f  = nn.Parameter(torch.empty(8,  IN).uniform_(-0.02, 0.02))
shared_16f = nn.Parameter(torch.empty(16, IN).uniform_(-0.02, 0.02))
shared_32f = nn.Parameter(torch.empty(32, IN).uniform_(-0.02, 0.02))
shared_64f = nn.Parameter(torch.empty(64, IN).uniform_(-0.02, 0.02))
shared_128f = nn.Parameter(torch.empty(128, IN).uniform_(-0.02, 0.02))

models = {
    'Linear':                (LinearCls(),              None,         OUT*IN,             0),
    'Tversky sigmoid 4f':    (TverskySigmoid(4),        None,         4*IN + OUT*IN,      4*IN),
    'Tversky sigmoid 8f':    (TverskySigmoid(8),        None,         8*IN + OUT*IN,      8*IN),
    'Tversky noFeatures':    (TverskyNoFeatures(),      None,         OUT*IN,             0),
    'TverskyNoFeat poly':    (TverskyNoFeatPoly(),      None,         OUT*IN,             0),
    'TverskyNoFeat scale':   (TverskyNoFeatLearnedScale(), None,      OUT*IN,             0),
    'Tversky lowrank r16':   (TverskyLowRank(16),       None,         OUT*16 + 16*IN,     0),
    'Tversky lowrank r32':   (TverskyLowRank(32),       None,         OUT*32 + 32*IN,     0),
    'Tversky shared 4f':     (TverskySharedCls(4,   shared_4f),   [shared_4f],   OUT*IN, 4*IN),
    'Tversky shared 8f':     (TverskySharedCls(8,   shared_8f),   [shared_8f],   OUT*IN, 8*IN),
    'Tversky shared 16f':    (TverskySharedCls(16,  shared_16f),  [shared_16f],  OUT*IN, 16*IN),
    'Tversky shared 32f':    (TverskySharedCls(32,  shared_32f),  [shared_32f],  OUT*IN, 32*IN),
    'Tversky shared 64f':    (TverskySharedCls(64,  shared_64f),  [shared_64f],  OUT*IN, 64*IN),
    'Tversky shared 128f':   (TverskySharedCls(128, shared_128f), [shared_128f], OUT*IN, 128*IN),
}

lin_time = None
print(f'{"Model":<26} {"Loss":>6} {"Acc":>6} {"A>B":>6} {"B>A":>6} {"Unr":>6} {"Avg":>6} {"ms":>6} {"StorMB(12L)":>12}')
print('-' * 96)

for name, (model, extra, proto_p, feat_p) in models.items():
    loss, acc, a0, a1, a2, avg = train_eval_model(model, extra)
    t = benchmark_model(model)
    if lin_time is None: lin_time = t

    stor = storage_mb(feat_p, proto_p, shared_feat=(feat_p > 0))
    print(f'{name:<26} {loss:>6.3f} {acc:>6.3f} {a0:>6.3f} {a1:>6.3f} {a2:>6.3f} {avg:>6.3f} {t:>5.2f}ms {stor:>10.3f}MB')

print(f'\nLinear storage ({N_LAYERS}L): {storage_linear():.3f}MB')

In [ ]:
# ============================================================
# Tversky variants — directional classification + LM task
# Standalone cell with MPS support
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# --- Config ---
DIM        = 768
VOCAB      = 8192
N_CONCEPTS = 800
N_TRAIN    = 20000
N_TEST     = 4000
N_EPOCHS   = 100
LR         = 1e-3
BATCH      = 512
N_LAYERS   = 12
OUT        = 3
IN         = DIM * 2

# --- Hierarchical concepts ---
concepts = torch.zeros(N_CONCEPTS, DIM)
feature_noise = 0.3
for i in range(N_CONCEPTS):
    level = i % 4
    if level == 0:
        active = torch.randperm(DIM)[:8]
        vals = torch.randn(len(active)).abs() + 1.0
    elif level == 1:
        parent = (i // 4) * 4
        parent_active = concepts[parent].nonzero().squeeze(-1)
        inherit = parent_active[torch.randperm(len(parent_active))[:6]]
        new = torch.randperm(DIM)[:12]
        active = torch.cat([inherit, new]).unique()
        vals = torch.randn(len(active)).abs() + 0.8
    elif level == 2:
        parent = min((i // 4) * 4 + 1, N_CONCEPTS - 1)
        parent_active = concepts[parent].nonzero().squeeze(-1)
        inherit = parent_active[torch.randperm(len(parent_active))[:min(10, len(parent_active))]]
        new = torch.randperm(DIM)[:16]
        active = torch.cat([inherit, new]).unique()
        vals = torch.randn(len(active)).abs() + 0.6
    else:
        parent = min((i // 4) * 4 + 2, N_CONCEPTS - 1)
        parent_active = concepts[parent].nonzero().squeeze(-1)
        inherit = parent_active[torch.randperm(len(parent_active))[:min(14, len(parent_active))]]
        new = torch.randperm(DIM)[:20]
        active = torch.cat([inherit, new]).unique()
        vals = torch.randn(len(active)).abs() + 0.4
    active = active.long().clamp(0, DIM - 1)
    concepts[i, active] = vals[:len(active)]
    concepts[i] += torch.randn(DIM) * feature_noise
concepts = F.normalize(concepts, dim=-1)

# Token embeddings for LM task
N_CLUSTERS = 64
cluster_ids = torch.arange(VOCAB) % N_CLUSTERS
freq_cpu = 1.0 / (torch.arange(1, VOCAB + 1).float() ** 0.8)
freq_cpu = freq_cpu / freq_cpu.sum()
token_embeddings = torch.zeros(VOCAB, DIM)
tokens_per_cluster = VOCAB // N_CLUSTERS
for c in range(N_CLUSTERS):
    s, e = c * tokens_per_cluster, min((c + 1) * tokens_per_cluster, VOCAB)
    center = torch.randn(DIM) * 2.0
    token_embeddings[s:e] = center + torch.randn(e - s, DIM) * 0.5
token_embeddings = F.normalize(token_embeddings, dim=-1)

# --- Build directional classification pairs ---
def make_dir_pairs(n):
    xs, ys, labels = [], [], []
    for _ in range(n):
        r = torch.randint(0, 3, (1,)).item()
        if r == 0:
            p = torch.randint(0, N_CONCEPTS // 4, (1,)).item() * 4
            c = min(p + torch.randint(1, 4, (1,)).item(), N_CONCEPTS - 1)
            xs.append(concepts[p]); ys.append(concepts[c]); labels.append(0)
        elif r == 1:
            p = torch.randint(0, N_CONCEPTS // 4, (1,)).item() * 4
            c = min(p + torch.randint(1, 4, (1,)).item(), N_CONCEPTS - 1)
            xs.append(concepts[c]); ys.append(concepts[p]); labels.append(1)
        else:
            i, j = torch.randint(0, N_CONCEPTS, (2,)).tolist()
            xs.append(concepts[i]); ys.append(concepts[j]); labels.append(2)
    return torch.stack(xs), torch.stack(ys), torch.tensor(labels)

train_xa, train_ya, train_labels = make_dir_pairs(N_TRAIN)
test_xa,  test_ya,  test_labels  = make_dir_pairs(N_TEST)
train_dir = torch.cat([train_xa, train_ya], dim=-1).to(device)
test_dir  = torch.cat([test_xa,  test_ya],  dim=-1).to(device)
train_labels = train_labels.to(device)
test_labels  = test_labels.to(device)

# --- Build LM next-token pairs ---
SEQ_LEN = 16
def make_lm_pairs(n):
    xs, labels = [], []
    for _ in range(n):
        seq = torch.multinomial(freq_cpu.expand(SEQ_LEN + 1, -1), 1).squeeze(-1)
        weights = torch.arange(1, SEQ_LEN + 1).float()
        weights = weights / weights.sum()
        ctx_weighted = (token_embeddings[seq[:-1]] * weights.unsqueeze(-1)).sum(0)
        x = torch.cat([token_embeddings[seq[-2]], ctx_weighted])
        labels.append(cluster_ids[seq[-1].item()].item() % OUT)
        xs.append(x)
    return torch.stack(xs), torch.tensor(labels)

train_lm_x, train_lm_y = make_lm_pairs(N_TRAIN)
test_lm_x,  test_lm_y  = make_lm_pairs(N_TEST)
train_lm_x = train_lm_x.to(device)
test_lm_x  = test_lm_x.to(device)
train_lm_y = train_lm_y.to(device)
test_lm_y  = test_lm_y.to(device)

print(f'Dir task label dist: {[(test_labels==i).sum().item() for i in range(OUT)]}')
print(f'LM  task label dist: {[(test_lm_y==i).sum().item() for i in range(OUT)]}')

# --- Storage helpers ---
def storage_mb(feat_params, proto_params, n_layers=N_LAYERS, shared_feat=True):
    feat_cost  = feat_params * 2 * (1 if shared_feat else n_layers)
    proto_cost = proto_params * 2 * n_layers
    return (feat_cost + proto_cost) / 1e6

def storage_linear():
    return OUT * IN * 2 * N_LAYERS / 1e6

# --- Models ---
class LinearCls(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.randn(OUT, IN) * 0.02)
    def forward(self, x): return x @ self.w.t()
    def param_count(self): return self.w.numel()

class TverskySigmoid(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.nf = nf
        self.features   = nn.Parameter(torch.empty(nf, IN).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f = x @ self.features.t()
        p_f = self.prototypes @ self.features.t()
        x_s = torch.sigmoid(x_f * 5.0)
        p_s = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.features.numel() + self.prototypes.numel() + 3

class TverskyNoFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f    = x @ self.prototypes.t()
        p_norm = F.normalize(self.prototypes, dim=-1)
        p_f    = p_norm @ p_norm.t()
        x_s    = torch.sigmoid(x_f * 5.0)
        p_s    = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3

class TverskyNoFeatPoly(nn.Module):
    def __init__(self):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        x_f    = x @ self.prototypes.t()
        p_norm = F.normalize(self.prototypes, dim=-1)
        p_f    = p_norm @ p_norm.t()
        x_s    = torch.clamp(x_f * 5.0 / 4 + 0.5, 0, 1)
        p_s    = torch.clamp(p_f * 5.0 / 4 + 0.5, 0, 1)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3

class TverskySharedCls(nn.Module):
    def __init__(self, nf, shared_feat):
        super().__init__()
        self.shared_feat = shared_feat
        self.nf = nf
        self.prototypes = nn.Parameter(torch.empty(OUT, IN).uniform_(-0.02, 0.02))
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        feat = self.shared_feat
        x_f  = x @ feat.t()
        p_f  = self.prototypes @ feat.t()
        x_s  = torch.sigmoid(x_f * 5.0)
        p_s  = torch.sigmoid(p_f * 5.0)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t*(x_a @ p_a.t()) - a*(x_a @ (1-p_s).t()) - b*((1-x_s) @ p_a.t())
    def param_count(self): return self.prototypes.numel() + 3

# --- Train/eval/benchmark ---
def train_eval(model, train_x, train_y, test_x, test_y, extra_params=None):
    model = model.to(device)
    params = list(model.parameters())
    if extra_params:
        params += [p for p in extra_params if not any(p is q for q in params)]
    opt = torch.optim.Adam(params, lr=LR)
    for _ in range(N_EPOCHS):
        idx = torch.randperm(len(train_x), device=device)
        for i in range(0, len(train_x), BATCH):
            b = idx[i:i+BATCH]
            opt.zero_grad()
            F.cross_entropy(model(train_x[b]), train_y[b]).backward()
            opt.step()
    with torch.no_grad():
        logits = model(test_x)
        loss   = F.cross_entropy(logits, test_y).item()
        preds  = logits.argmax(-1)
        acc    = (preds == test_y).float().mean().item()
        per_class = [(preds[test_y==i] == i).float().mean().item() if (test_y==i).any() else 0.0 for i in range(OUT)]
        avg    = sum(per_class) / OUT
    return loss, acc, per_class, avg

def benchmark(model, test_x, n_runs=500):
    model = model.to(device)
    for _ in range(50): model(test_x)
    if device.type == 'mps': torch.mps.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_runs): model(test_x)
    if device.type == 'mps': torch.mps.synchronize()
    return (time.perf_counter() - t0) / n_runs * 1000

# --- Shared feature banks ---
def make_shared(nf): return nn.Parameter(torch.empty(nf, IN).uniform_(-0.02, 0.02)).to(device)
sf = {nf: make_shared(nf) for nf in [4, 8, 16, 32, 64, 128]}

model_configs = {
    'Linear':              (LinearCls(),                        None,              OUT*IN, 0),
    'Tversky sigmoid 4f':  (TverskySigmoid(4),                  None,              4*IN+OUT*IN, 4*IN),
    'Tversky sigmoid 8f':  (TverskySigmoid(8),                  None,              8*IN+OUT*IN, 8*IN),
    'Tversky noFeatures':  (TverskyNoFeatures(),                None,              OUT*IN, 0),
    'TverskyNoFeat poly':  (TverskyNoFeatPoly(),                None,              OUT*IN, 0),
    'Tversky shared 4f':   (TverskySharedCls(4,   sf[4]),       [sf[4]],           OUT*IN, 4*IN),
    'Tversky shared 8f':   (TverskySharedCls(8,   sf[8]),       [sf[8]],           OUT*IN, 8*IN),
    'Tversky shared 16f':  (TverskySharedCls(16,  sf[16]),      [sf[16]],          OUT*IN, 16*IN),
    'Tversky shared 32f':  (TverskySharedCls(32,  sf[32]),      [sf[32]],          OUT*IN, 32*IN),
    'Tversky shared 64f':  (TverskySharedCls(64,  sf[64]),      [sf[64]],          OUT*IN, 64*IN),
    'Tversky shared 128f': (TverskySharedCls(128, sf[128]),     [sf[128]],         OUT*IN, 128*IN),
}

for task_name, train_x, train_y, test_x, test_y in [
    ('DIRECTIONAL CLASSIFICATION', train_dir, train_labels, test_dir, test_labels),
    ('LM NEXT-TOKEN PREDICTION',   train_lm_x, train_lm_y, test_lm_x, test_lm_y),
]:
    print(f'\n=== {task_name} ===')
    print(f'{"Model":<26} {"Loss":>6} {"Acc":>6} {"Avg":>6} {"ms":>6} {"StorMB(12L)":>12}')
    print('-' * 70)

    sf = {nf: nn.Parameter(torch.empty(nf, IN).uniform_(-0.02, 0.02)).to(device) for nf in [4, 8, 16, 32, 64, 128]}

    model_configs = {
        'Linear':              (lambda: LinearCls(),                              None,          OUT*IN, 0),
        'Tversky sigmoid 4f':  (lambda: TverskySigmoid(4),                        None,          4*IN+OUT*IN, 4*IN),
        'Tversky sigmoid 8f':  (lambda: TverskySigmoid(8),                        None,          8*IN+OUT*IN, 8*IN),
        'Tversky noFeatures':  (lambda: TverskyNoFeatures(),                      None,          OUT*IN, 0),
        'TverskyNoFeat poly':  (lambda: TverskyNoFeatPoly(),                      None,          OUT*IN, 0),
        'Tversky shared 4f':   (lambda: (lambda f: TverskySharedCls(4,   f))(nn.Parameter(torch.empty(4,   IN, device=device).uniform_(-0.02, 0.02))), None, OUT*IN, 4*IN),
        'Tversky shared 8f':   (lambda: (lambda f: TverskySharedCls(8,   f))(nn.Parameter(torch.empty(8,   IN, device=device).uniform_(-0.02, 0.02))), None, OUT*IN, 8*IN),
        'Tversky shared 16f':  (lambda: (lambda f: TverskySharedCls(16,  f))(nn.Parameter(torch.empty(16,  IN, device=device).uniform_(-0.02, 0.02))), None, OUT*IN, 16*IN),
        'Tversky shared 32f':  (lambda: (lambda f: TverskySharedCls(32,  f))(nn.Parameter(torch.empty(32,  IN, device=device).uniform_(-0.02, 0.02))), None, OUT*IN, 32*IN),
        'Tversky shared 64f':  (lambda: (lambda f: TverskySharedCls(64,  f))(nn.Parameter(torch.empty(64,  IN, device=device).uniform_(-0.02, 0.02))), None, OUT*IN, 64*IN),
        'Tversky shared 128f': (lambda: (lambda f: TverskySharedCls(128, f))(nn.Parameter(torch.empty(128, IN, device=device).uniform_(-0.02, 0.02))), None, OUT*IN, 128*IN),
    }

    for name, (factory, _, proto_p, feat_p) in model_configs.items():
        model = factory()
        extra = [model.shared_feat] if hasattr(model, 'shared_feat') else None
        loss, acc, per_class, avg = train_eval(model, train_x, train_y, test_x, test_y, extra)
        t = benchmark(model, test_x)
        stor = storage_mb(feat_p, proto_p, shared_feat=(feat_p > 0))
        print(f'{name:<26} {loss:>6.3f} {acc:>6.3f} {avg:>6.3f} {t:>5.2f}ms {stor:>10.3f}MB')

print(f'\nLinear storage ({N_LAYERS}L): {storage_linear():.3f}MB')

In [3]:
"""
Alternative Linear Layer Exploration
=====================================
Goal: Find a linear layer variant that is BETTER, FASTER, or SMALLER
than standard linear — or any beneficial combination.

Context: Ternary quantization (1.6 bits/param), 16MB budget, H100 GPUs.
Each saved param = room for more layers. Each saved ms = more training steps.

Candidates:
1. Standard Linear (baseline)
2. Low-Rank Factored (two smaller matrices)
3. Block-Diagonal (independent blocks along diagonal)
4. Kronecker Product (factored as A ⊗ B)
5. Shared-Weight / Hash-Trick (fewer unique params)
6. Monarch Matrix (structured butterfly factorization)
7. Grouped Linear (like grouped convolution)
8. Sparse Block (dense blocks with fixed zero structure)

Each is tested on:
- Task A: Directional classification (asymmetric relationships)
- Task B: LM-style next-token prediction (the actual target)

Metrics: accuracy, loss, speed (ms/forward), storage (param count),
         and storage * speed product (lower = better)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import math

torch.manual_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# Config — matches real model dimensions
# ============================================================
DIM = 768           # model_dim
OUT_DIM = 768       # output dim (same for attn proj, different for MLP)
MLP_HIDDEN = 1536   # MLP_MULT=2 * 768
VOCAB = 8192
N_LAYERS = 12
N_CONCEPTS = 500
N_TRAIN = 10000
N_TEST = 2000
N_EPOCHS = 80
LR = 1e-3
BATCH = 512
OUT_CLASSES = 3     # for classification tasks

# ============================================================
# Synthetic Data
# ============================================================

# --- Hierarchical concepts for directional task ---
concepts = torch.zeros(N_CONCEPTS, DIM)
for i in range(N_CONCEPTS):
    level = i % 3
    if level == 0:
        active = torch.randperm(DIM)[:16]
    elif level == 1:
        parent = (i // 3) * 3
        parent_active = concepts[parent].nonzero().squeeze(-1)
        new = torch.randperm(DIM)[:12]
        active = torch.cat([parent_active, new])[:28]
    else:
        parent = ((i // 3) * 3) + 1
        if parent < N_CONCEPTS:
            parent_active = concepts[parent].nonzero().squeeze(-1)
        else:
            parent_active = torch.tensor([])
        new = torch.randperm(DIM)[:10]
        active = torch.cat([parent_active.flatten(), new])[:40]
    active = active.long().clamp(0, DIM - 1).unique()
    concepts[i, active] = torch.randn(len(active)).abs() + 0.5
concepts = F.normalize(concepts, dim=-1)

def make_dir_pairs(n):
    xs, ys, labels = [], [], []
    for _ in range(n):
        r = torch.randint(0, 3, (1,)).item()
        if r == 0:
            p = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            c = min(p + torch.randint(1, 3, (1,)).item(), N_CONCEPTS - 1)
            xs.append(concepts[p]); ys.append(concepts[c]); labels.append(0)
        elif r == 1:
            p = torch.randint(0, N_CONCEPTS // 3, (1,)).item() * 3
            c = min(p + torch.randint(1, 3, (1,)).item(), N_CONCEPTS - 1)
            xs.append(concepts[c]); ys.append(concepts[p]); labels.append(1)
        else:
            i, j = torch.randint(0, N_CONCEPTS, (2,)).tolist()
            xs.append(concepts[i]); ys.append(concepts[j]); labels.append(2)
    return torch.stack(xs), torch.stack(ys), torch.tensor(labels)

# Directional task: input = concat(A, B), predict relationship
train_xa, train_ya, train_labels = make_dir_pairs(N_TRAIN)
test_xa, test_ya, test_labels = make_dir_pairs(N_TEST)
DIR_IN = DIM * 2  # concatenated pair
train_dir = torch.cat([train_xa, train_ya], dim=-1).to(device)
test_dir = torch.cat([test_xa, test_ya], dim=-1).to(device)
train_labels = train_labels.to(device)
test_labels = test_labels.to(device)

# --- LM-style task: predict class from context embedding ---
freq_cpu = 1.0 / (torch.arange(1, VOCAB + 1).float() ** 0.8)
freq_cpu = freq_cpu / freq_cpu.sum()
token_embeddings = torch.randn(VOCAB, DIM)
N_CLUSTERS = 64
cluster_ids = torch.arange(VOCAB) % N_CLUSTERS
for c in range(N_CLUSTERS):
    s, e = c * (VOCAB // N_CLUSTERS), min((c + 1) * (VOCAB // N_CLUSTERS), VOCAB)
    center = torch.randn(DIM) * 2.0
    token_embeddings[s:e] = center + torch.randn(e - s, DIM) * 0.5
token_embeddings = F.normalize(token_embeddings, dim=-1)

SEQ_LEN = 16
def make_lm_pairs(n):
    xs, labels = [], []
    for _ in range(n):
        seq = torch.multinomial(freq_cpu.expand(SEQ_LEN + 1, -1), 1).squeeze(-1)
        weights = torch.arange(1, SEQ_LEN + 1).float()
        weights = weights / weights.sum()
        ctx = (token_embeddings[seq[:-1]] * weights.unsqueeze(-1)).sum(0)
        xs.append(ctx)
        labels.append(cluster_ids[seq[-1].item()].item() % OUT_CLASSES)
    return torch.stack(xs), torch.tensor(labels)

train_lm_x, train_lm_y = make_lm_pairs(N_TRAIN)
test_lm_x, test_lm_y = make_lm_pairs(N_TEST)
train_lm_x, test_lm_x = train_lm_x.to(device), test_lm_x.to(device)
train_lm_y, test_lm_y = train_lm_y.to(device), test_lm_y.to(device)

print(f'Dir labels: {[(test_labels==i).sum().item() for i in range(OUT_CLASSES)]}')
print(f'LM  labels: {[(test_lm_y==i).sum().item() for i in range(OUT_CLASSES)]}')

# ============================================================
# Layer Implementations
# ============================================================

class StandardLinear(nn.Module):
    """Baseline: full dense linear."""
    def __init__(self, in_f, out_f):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
    def forward(self, x):
        return x @ self.w.t()
    def param_count(self):
        return self.w.numel()
    def ternary_bytes(self):
        return self.w.numel() * 1.6 / 8  # 1.6 bits/param


class LowRankLinear(nn.Module):
    """Factored: W ≈ A @ B where A=[out,r], B=[r,in]. Rank r controls compression."""
    def __init__(self, in_f, out_f, rank=64):
        super().__init__()
        self.A = nn.Parameter(torch.randn(out_f, rank) * 0.02)
        self.B = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.rank = rank
    def forward(self, x):
        return (x @ self.B.t()) @ self.A.t()
    def param_count(self):
        return self.A.numel() + self.B.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class BlockDiagonalLinear(nn.Module):
    """Block-diagonal: n_blocks independent [block_out, block_in] matrices.
    Requires in_f and out_f divisible by n_blocks.
    Params = in_f * out_f / n_blocks. Speed: n_blocks small matmuls (can be batched)."""
    def __init__(self, in_f, out_f, n_blocks=8):
        super().__init__()
        assert in_f % n_blocks == 0 and out_f % n_blocks == 0
        self.n_blocks = n_blocks
        self.block_in = in_f // n_blocks
        self.block_out = out_f // n_blocks
        # [n_blocks, block_out, block_in] — batched matmul
        self.blocks = nn.Parameter(torch.randn(n_blocks, self.block_out, self.block_in) * 0.02)
    def forward(self, x):
        bsz = x.shape[:-1]
        # Reshape to [..., n_blocks, block_in]
        x_blocks = x.reshape(*bsz, self.n_blocks, self.block_in)
        # Einstein: ...bi, bio -> ...bo
        out = torch.einsum('...bi,bio->...bo', x_blocks, self.blocks)
        return out.reshape(*bsz, self.n_blocks * self.block_out)
    def param_count(self):
        return self.blocks.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class BlockDiagPlusMix(nn.Module):
    """Block-diagonal + cheap cross-block mixing via a small shared matrix.
    Addresses the main weakness of block-diagonal (no cross-block interaction)
    with minimal parameter overhead."""
    def __init__(self, in_f, out_f, n_blocks=8, mix_rank=16):
        super().__init__()
        assert in_f % n_blocks == 0 and out_f % n_blocks == 0
        self.n_blocks = n_blocks
        self.block_in = in_f // n_blocks
        self.block_out = out_f // n_blocks
        self.blocks = nn.Parameter(torch.randn(n_blocks, self.block_out, self.block_in) * 0.02)
        # Cross-block mixing: low-rank [out_f, in_f] via two small matrices
        self.mix_down = nn.Parameter(torch.randn(mix_rank, in_f) * 0.02)
        self.mix_up = nn.Parameter(torch.randn(out_f, mix_rank) * 0.02)
    def forward(self, x):
        bsz = x.shape[:-1]
        # Block-diagonal path
        x_blocks = x.reshape(*bsz, self.n_blocks, self.block_in)
        block_out = torch.einsum('...bi,bio->...bo', x_blocks, self.blocks)
        block_out = block_out.reshape(*bsz, self.n_blocks * self.block_out)
        # Low-rank cross-block mixing
        mix_out = (x @ self.mix_down.t()) @ self.mix_up.t()
        return block_out + mix_out
    def param_count(self):
        return self.blocks.numel() + self.mix_down.numel() + self.mix_up.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class GroupedLinear(nn.Module):
    """Grouped linear: like grouped convolution. Input and output split into g groups,
    each with independent weights. Params = in_f * out_f / g."""
    def __init__(self, in_f, out_f, groups=4):
        super().__init__()
        assert in_f % groups == 0 and out_f % groups == 0
        self.groups = groups
        self.group_in = in_f // groups
        self.group_out = out_f // groups
        self.weight = nn.Parameter(torch.randn(groups, self.group_out, self.group_in) * 0.02)
    def forward(self, x):
        bsz = x.shape[:-1]
        x_g = x.reshape(*bsz, self.groups, self.group_in)
        out = torch.einsum('...gi,goi->...go', x_g, self.weight)
        return out.reshape(*bsz, self.groups * self.group_out)
    def param_count(self):
        return self.weight.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class KroneckerLinear(nn.Module):
    """Kronecker product: W = A ⊗ B. If A=[a1,a2], B=[b1,b2], then W=[a1*b1, a2*b2].
    Params = a1*a2 + b1*b2 instead of (a1*b1)*(a2*b2).
    Requires in_f = a2*b2, out_f = a1*b1."""
    def __init__(self, in_f, out_f, a_rows=None, a_cols=None):
        super().__init__()
        # Try to find good factorization
        if a_rows is None:
            # Find factor of out_f closest to sqrt(out_f)
            a_rows = int(math.sqrt(out_f))
            while out_f % a_rows != 0:
                a_rows -= 1
        if a_cols is None:
            a_cols = int(math.sqrt(in_f))
            while in_f % a_cols != 0:
                a_cols -= 1
        self.a_rows, self.a_cols = a_rows, a_cols
        self.b_rows = out_f // a_rows
        self.b_cols = in_f // a_cols
        self.A = nn.Parameter(torch.randn(a_rows, a_cols) * 0.02)
        self.B = nn.Parameter(torch.randn(self.b_rows, self.b_cols) * 0.02)
        self.out_f = out_f
        self.in_f = in_f
    def forward(self, x):
        # Efficient Kronecker: reshape input, multiply by B, reshape, multiply by A
        bsz = x.shape[:-1]
        # x: [..., a_cols * b_cols] -> [..., a_cols, b_cols]
        x_r = x.reshape(*bsz, self.a_cols, self.b_cols)
        # Multiply by B: [..., a_cols, b_rows]
        xB = x_r @ self.B.t()
        # Transpose and multiply by A: [..., b_rows, a_rows]
        xBA = xB.transpose(-2, -1) @ self.A.t()
        # Reshape to [..., a_rows * b_rows]
        return xBA.transpose(-2, -1).reshape(*bsz, self.out_f)
    def param_count(self):
        return self.A.numel() + self.B.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class HashLinear(nn.Module):
    """Hash-trick weight sharing: n_buckets unique params, virtual [out, in] mapped via hash.
    Compression ratio = (out*in) / n_buckets."""
    def __init__(self, in_f, out_f, n_buckets=65536):
        super().__init__()
        self.in_f, self.out_f = in_f, out_f
        self.n_buckets = n_buckets
        self.table = nn.Parameter(torch.randn(n_buckets) * 0.02)
        # Precompute hash indices (vectorized)
        rows = torch.arange(out_f).unsqueeze(1).expand(out_f, in_f)
        cols = torch.arange(in_f).unsqueeze(0).expand(out_f, in_f)
        idx = ((rows * 1000003 + cols * 999983) ^ (rows * 31 + cols * 7)) % n_buckets
        self.register_buffer('hash_idx', idx.long())
    def forward(self, x):
        W = self.table[self.hash_idx]
        return x @ W.t()
    def param_count(self):
        return self.n_buckets  # only unique params matter
    def ternary_bytes(self):
        return self.n_buckets * 1.6 / 8


class StructuredSparse(nn.Module):
    """Structured sparsity: full weight matrix with a fixed binary mask.
    Mask keeps top-k% of positions (random but fixed). Training only updates
    unmasked positions. Storage = sparsity_ratio * full params."""
    def __init__(self, in_f, out_f, keep_ratio=0.25):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
        mask = torch.zeros(out_f, in_f, dtype=torch.bool)
        n_keep = int(out_f * in_f * keep_ratio)
        indices = torch.randperm(out_f * in_f)[:n_keep]
        mask.view(-1)[indices] = True
        self.register_buffer('mask', mask)
        self.keep_ratio = keep_ratio
    def forward(self, x):
        w = self.weight * self.mask
        return x @ w.t()
    def param_count(self):
        return int(self.mask.sum().item())
    def ternary_bytes(self):
        # Need to store: mask (1 bit/position) + values (1.6 bits/nonzero)
        total_pos = self.weight.numel()
        n_nonzero = self.param_count()
        mask_bytes = total_pos / 8
        value_bytes = n_nonzero * 1.6 / 8
        return mask_bytes + value_bytes


class DualPathLinear(nn.Module):
    """Two parallel paths: a small dense matrix + block-diagonal.
    Dense handles global patterns, block-diagonal handles local."""
    def __init__(self, in_f, out_f, dense_rank=32, n_blocks=8):
        super().__init__()
        self.dense_down = nn.Parameter(torch.randn(dense_rank, in_f) * 0.02)
        self.dense_up = nn.Parameter(torch.randn(out_f, dense_rank) * 0.02)
        assert in_f % n_blocks == 0 and out_f % n_blocks == 0
        self.n_blocks = n_blocks
        self.block_in = in_f // n_blocks
        self.block_out = out_f // n_blocks
        self.blocks = nn.Parameter(torch.randn(n_blocks, self.block_out, self.block_in) * 0.02)
    def forward(self, x):
        bsz = x.shape[:-1]
        # Dense global path
        dense = (x @ self.dense_down.t()) @ self.dense_up.t()
        # Block-diagonal local path
        x_b = x.reshape(*bsz, self.n_blocks, self.block_in)
        block = torch.einsum('...bi,bio->...bo', x_b, self.blocks)
        block = block.reshape(*bsz, self.n_blocks * self.block_out)
        return dense + block
    def param_count(self):
        return self.dense_down.numel() + self.dense_up.numel() + self.blocks.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class ButterFlyLinear(nn.Module):
    """Butterfly factorization: log2(n) stages of structured sparse matmuls.
    Total params = O(n * log(n)) vs O(n^2) for dense.
    Requires in_f == out_f and power of 2."""
    def __init__(self, dim):
        super().__init__()
        assert dim > 0 and (dim & (dim - 1)) == 0, "dim must be power of 2"
        self.dim = dim
        self.n_stages = int(math.log2(dim))
        # Each stage has dim/2 2x2 butterfly matrices = dim params
        self.factors = nn.ParameterList([
            nn.Parameter(torch.randn(dim, 2) * 0.02) for _ in range(self.n_stages)
        ])
    def forward(self, x):
        bsz = x.shape[:-1]
        y = x.reshape(*bsz, self.dim)
        half = self.dim // 2
        for stage in range(self.n_stages):
            stride = 1 << stage
            # Split into pairs based on bit pattern
            idx0 = torch.arange(self.dim, device=x.device)
            block_idx = idx0 // (2 * stride)
            within_block = idx0 % (2 * stride)
            is_second = (within_block >= stride).long()
            pair_idx = block_idx * (2 * stride) + within_block % stride

            f = self.factors[stage]  # [dim, 2]
            a, b = f[:, 0], f[:, 1]

            y_new = y.clone()
            even_mask = is_second == 0
            odd_mask = is_second == 1
            partner = pair_idx + stride * (1 - 2 * is_second)
            partner = partner.clamp(0, self.dim - 1)

            y_new[..., even_mask] = a[even_mask] * y[..., even_mask] + b[even_mask] * y[..., partner[even_mask]]
            y_new[..., odd_mask] = a[odd_mask] * y[..., odd_mask] + b[odd_mask] * y[..., partner[odd_mask]]
            y = y_new
        return y
    def param_count(self):
        return self.dim * 2 * self.n_stages
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


# ============================================================
# Training + Evaluation
# ============================================================

def train_eval(model, train_x, train_y, test_x, test_y, epochs=N_EPOCHS):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for _ in range(epochs):
        idx = torch.randperm(len(train_x), device=device)
        for i in range(0, len(train_x), BATCH):
            b = idx[i:i + BATCH]
            opt.zero_grad()
            F.cross_entropy(model(train_x[b]), train_y[b]).backward()
            opt.step()
    with torch.no_grad():
        logits = model(test_x)
        loss = F.cross_entropy(logits, test_y).item()
        acc = (logits.argmax(-1) == test_y).float().mean().item()
        per_class = [(logits[test_y == i].argmax(-1) == i).float().mean().item()
                     if (test_y == i).any() else 0.0 for i in range(OUT_CLASSES)]
        avg = sum(per_class) / OUT_CLASSES
    return loss, acc, avg

def benchmark(model, test_x, n_warmup=50, n_runs=500):
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        for _ in range(n_warmup):
            model(test_x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_runs):
            model(test_x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_runs * 1000

def storage_per_layer(param_count):
    """Ternary storage in MB for one layer."""
    return param_count * 1.6 / 8 / 1e6

def storage_total(param_count, n_layers=N_LAYERS):
    """Total ternary storage in MB across all layers."""
    return storage_per_layer(param_count) * n_layers


# ============================================================
# Build models for classification (in=DIR_IN, out=OUT_CLASSES)
# ============================================================

def build_dir_models():
    """Build classification models: maps DIR_IN -> OUT_CLASSES."""
    return {
        'Linear':                StandardLinear(DIR_IN, OUT_CLASSES),
        'LowRank r=8':          LowRankLinear(DIR_IN, OUT_CLASSES, rank=8),
        'LowRank r=32':         LowRankLinear(DIR_IN, OUT_CLASSES, rank=32),
        'LowRank r=128':        LowRankLinear(DIR_IN, OUT_CLASSES, rank=128),
        'BlockDiag b=4':        BlockDiagonalLinear(DIR_IN, OUT_CLASSES, n_blocks=1),  # can't split 3 classes into blocks; test on proj-style
        'Kronecker':            KroneckerLinear(DIR_IN, OUT_CLASSES, a_rows=1, a_cols=48),  # 1x48 ⊗ 3x32
        'Hash 4096':            HashLinear(DIR_IN, OUT_CLASSES, n_buckets=4096),
        'Sparse 25%':           StructuredSparse(DIR_IN, OUT_CLASSES, keep_ratio=0.25),
        'Sparse 50%':           StructuredSparse(DIR_IN, OUT_CLASSES, keep_ratio=0.50),
    }


# ============================================================
# Build projection models (in=DIM, out=DIM) — the real target
# This simulates the attn output projection or MLP projection
# ============================================================

def build_proj_models():
    """Build projection models: maps DIM -> DIM. This is the actual use case."""
    models = {
        'Linear':                 StandardLinear(DIM, DIM),
        'LowRank r=64':          LowRankLinear(DIM, DIM, rank=64),
        'LowRank r=128':         LowRankLinear(DIM, DIM, rank=128),
        'LowRank r=256':         LowRankLinear(DIM, DIM, rank=256),
        'LowRank r=384':         LowRankLinear(DIM, DIM, rank=384),
        'BlockDiag b=2':         BlockDiagonalLinear(DIM, DIM, n_blocks=2),
        'BlockDiag b=4':         BlockDiagonalLinear(DIM, DIM, n_blocks=4),
        'BlockDiag b=8':         BlockDiagonalLinear(DIM, DIM, n_blocks=8),
        'BlockDiag b=16':        BlockDiagonalLinear(DIM, DIM, n_blocks=16),
        'BD4 + mix16':           BlockDiagPlusMix(DIM, DIM, n_blocks=4, mix_rank=16),
        'BD4 + mix32':           BlockDiagPlusMix(DIM, DIM, n_blocks=4, mix_rank=32),
        'BD8 + mix16':           BlockDiagPlusMix(DIM, DIM, n_blocks=8, mix_rank=16),
        'BD8 + mix32':           BlockDiagPlusMix(DIM, DIM, n_blocks=8, mix_rank=32),
        'Grouped g=2':           GroupedLinear(DIM, DIM, groups=2),
        'Grouped g=4':           GroupedLinear(DIM, DIM, groups=4),
        'Kronecker':             KroneckerLinear(DIM, DIM),
        'DualPath r32 b8':       DualPathLinear(DIM, DIM, dense_rank=32, n_blocks=8),
        'DualPath r64 b4':       DualPathLinear(DIM, DIM, dense_rank=64, n_blocks=4),
        'Sparse 25%':            StructuredSparse(DIM, DIM, keep_ratio=0.25),
        'Sparse 50%':            StructuredSparse(DIM, DIM, keep_ratio=0.50),
        'Hash 65536':            HashLinear(DIM, DIM, n_buckets=65536),
        'Hash 131072':           HashLinear(DIM, DIM, n_buckets=131072),
    }
    # Butterfly only works for power-of-2 (768 is not, skip)
    return models


# ============================================================
# Run projection benchmark (speed + storage only, no task accuracy)
# ============================================================

print('\n' + '=' * 100)
print('PROJECTION BENCHMARK (DIM -> DIM): Speed + Storage')
print('=' * 100)
print(f'{"Model":<26} {"Params":>10} {"1L MB":>8} {"12L MB":>8} {"ms":>8} {"vs Lin":>8} {"Efficiency":>10}')
print('-' * 90)

# Create test data for projection benchmark
proj_test = torch.randn(2000, DIM, device=device)

proj_models = build_proj_models()
lin_speed = None
lin_params = None

for name, model in proj_models.items():
    pc = model.param_count()
    mb1 = storage_per_layer(pc)
    mb12 = storage_total(pc)
    t = benchmark(model, proj_test)
    if lin_speed is None:
        lin_speed = t
        lin_params = pc
    speedup = f'{t/lin_speed:.2f}x'
    compression = f'{pc/lin_params:.1%}'
    # Efficiency = quality proxy * speed. Lower storage + lower time = better
    efficiency = f'{mb12 * t:.2f}'
    print(f'{name:<26} {pc:>10,} {mb1:>7.3f}M {mb12:>7.2f}M {t:>7.2f}ms {speedup:>8} {efficiency:>10}')


# ============================================================
# Run classification task (accuracy + speed + storage)
# ============================================================

print('\n' + '=' * 100)
print('DIRECTIONAL CLASSIFICATION (DIR_IN -> 3 classes)')
print('=' * 100)
print(f'{"Model":<26} {"Loss":>6} {"Acc":>6} {"Avg":>6} {"Params":>10} {"12L MB":>8} {"ms":>8}')
print('-' * 85)

dir_models = {
    'Linear':          StandardLinear(DIR_IN, OUT_CLASSES),
    'LowRank r=32':    LowRankLinear(DIR_IN, OUT_CLASSES, rank=32),
    'LowRank r=128':   LowRankLinear(DIR_IN, OUT_CLASSES, rank=128),
    'Sparse 25%':      StructuredSparse(DIR_IN, OUT_CLASSES, keep_ratio=0.25),
    'Sparse 50%':      StructuredSparse(DIR_IN, OUT_CLASSES, keep_ratio=0.50),
    'Hash 1024':       HashLinear(DIR_IN, OUT_CLASSES, n_buckets=1024),
    'Hash 4096':       HashLinear(DIR_IN, OUT_CLASSES, n_buckets=4096),
}

for name, model in dir_models.items():
    loss, acc, avg = train_eval(model, train_dir, train_labels, test_dir, test_labels)
    t = benchmark(model, test_dir)
    pc = model.param_count()
    mb = storage_total(pc)
    print(f'{name:<26} {loss:>6.3f} {acc:>6.3f} {avg:>6.3f} {pc:>10,} {mb:>7.2f}M {t:>7.2f}ms')


# ============================================================
# Run LM task (accuracy on projection-sized models)
# For this we test DIM -> OUT_CLASSES through a projection layer
# ============================================================

print('\n' + '=' * 100)
print('LM NEXT-TOKEN PREDICTION (DIM -> 3 classes)')
print('=' * 100)
print(f'{"Model":<26} {"Loss":>6} {"Acc":>6} {"Avg":>6} {"Params":>10} {"12L MB":>8} {"ms":>8}')
print('-' * 85)

lm_models = {
    'Linear':          StandardLinear(DIM, OUT_CLASSES),
    'LowRank r=32':    LowRankLinear(DIM, OUT_CLASSES, rank=32),
    'LowRank r=128':   LowRankLinear(DIM, OUT_CLASSES, rank=128),
    'Sparse 25%':      StructuredSparse(DIM, OUT_CLASSES, keep_ratio=0.25),
    'Sparse 50%':      StructuredSparse(DIM, OUT_CLASSES, keep_ratio=0.50),
    'Hash 512':        HashLinear(DIM, OUT_CLASSES, n_buckets=512),
    'Hash 2048':       HashLinear(DIM, OUT_CLASSES, n_buckets=2048),
}

for name, model in lm_models.items():
    loss, acc, avg = train_eval(model, train_lm_x, train_lm_y, test_lm_x, test_lm_y)
    t = benchmark(model, test_lm_x)
    pc = model.param_count()
    mb = storage_total(pc)
    print(f'{name:<26} {loss:>6.3f} {acc:>6.3f} {avg:>6.3f} {pc:>10,} {mb:>7.2f}M {t:>7.2f}ms')

Device: cuda
Dir labels: [673, 632, 695]
LM  labels: [740, 641, 619]

PROJECTION BENCHMARK (DIM -> DIM): Speed + Storage
Model                          Params    1L MB   12L MB       ms   vs Lin Efficiency
------------------------------------------------------------------------------------------
Linear                        589,824   0.118M    1.42M    0.07ms    1.00x       0.10
LowRank r=64                   98,304   0.020M    0.24M    0.03ms    0.44x       0.01
LowRank r=128                 196,608   0.039M    0.47M    0.04ms    0.58x       0.02
LowRank r=256                 393,216   0.079M    0.94M    0.06ms    0.88x       0.06
LowRank r=384                 589,824   0.118M    1.42M    0.08ms    1.19x       0.12
BlockDiag b=2                 294,912   0.059M    0.71M    0.04ms    0.62x       0.03
BlockDiag b=4                 147,456   0.029M    0.35M    0.03ms    0.40x       0.01
BlockDiag b=8                  73,728   0.015M    0.18M    0.03ms    0.37x       0.00
BlockDiag b=16 

In [5]:
"""
Mini-Transformer: test linear variants in real transformer context.
2-4 layer transformer trained on synthetic next-token prediction.
Measures: val_loss, ms/step, params, to find Pareto-optimal layers.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import math

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# Config
# ============================================================
DIM = 768
NUM_HEADS = 8
NUM_KV_HEADS = 4
HEAD_DIM = DIM // NUM_HEADS  # 96
MLP_MULT = 2
HIDDEN = DIM * MLP_MULT
VOCAB = 512         # small vocab for fast iteration
SEQ_LEN = 128
BATCH = 32
N_LAYERS = 3
N_STEPS = 400
EVAL_EVERY = 100
LR = 3e-4

# ============================================================
# Synthetic data: Markov-ish sequences with structure
# ============================================================
# Create token transition probabilities with cluster structure
N_CLUSTERS = 16
tokens_per_cluster = VOCAB // N_CLUSTERS
transition = torch.zeros(VOCAB, VOCAB)
for c in range(N_CLUSTERS):
    s, e = c * tokens_per_cluster, (c + 1) * tokens_per_cluster
    # High prob within cluster
    transition[s:e, s:e] = 1.0
    # Lower prob to next cluster (creates sequential patterns)
    next_c = ((c + 1) % N_CLUSTERS)
    ns, ne = next_c * tokens_per_cluster, (next_c + 1) * tokens_per_cluster
    transition[s:e, ns:ne] = 0.3
    # Small prob to random cluster
    rand_c = ((c + 3) % N_CLUSTERS)
    rs, re = rand_c * tokens_per_cluster, (rand_c + 1) * tokens_per_cluster
    transition[s:e, rs:re] = 0.1
# Normalize rows
transition = transition / transition.sum(dim=-1, keepdim=True)

def generate_sequences(n_seqs, seq_len):
    seqs = torch.zeros(n_seqs, seq_len + 1, dtype=torch.long)
    seqs[:, 0] = torch.randint(0, VOCAB, (n_seqs,))
    for t in range(seq_len):
        probs = transition[seqs[:, t]]
        seqs[:, t + 1] = torch.multinomial(probs, 1).squeeze(-1)
    return seqs

train_data = generate_sequences(2000, SEQ_LEN).to(device)
val_data = generate_sequences(500, SEQ_LEN).to(device)

# ============================================================
# Linear layer variants (only the promising ones from notebook 1)
# ============================================================

class StandardLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
    def forward(self, x):
        return F.linear(x, self.weight)
    def param_count(self):
        return self.weight.numel()

class GroupedLinear(nn.Module):
    """Groups=g: input and output split into g independent groups."""
    def __init__(self, in_f, out_f, groups=2):
        super().__init__()
        assert in_f % groups == 0 and out_f % groups == 0
        self.groups = groups
        self.group_in = in_f // groups
        self.group_out = out_f // groups
        self.weight = nn.Parameter(torch.randn(groups, self.group_out, self.group_in) * 0.02)
    def forward(self, x):
        bsz = x.shape[:-1]
        x_g = x.reshape(*bsz, self.groups, self.group_in)
        out = torch.einsum('...gi,goi->...go', x_g, self.weight)
        return out.reshape(*bsz, self.groups * self.group_out)
    def param_count(self):
        return self.weight.numel()

class BlockDiagPlusMix(nn.Module):
    """Block-diagonal + low-rank cross-block mixing."""
    def __init__(self, in_f, out_f, n_blocks=4, mix_rank=32):
        super().__init__()
        assert in_f % n_blocks == 0 and out_f % n_blocks == 0
        self.n_blocks = n_blocks
        self.block_in = in_f // n_blocks
        self.block_out = out_f // n_blocks
        self.blocks = nn.Parameter(torch.randn(n_blocks, self.block_out, self.block_in) * 0.02)
        self.mix_down = nn.Parameter(torch.randn(mix_rank, in_f) * 0.02)
        self.mix_up = nn.Parameter(torch.randn(out_f, mix_rank) * 0.02)

    def forward(self, x):
        bsz = x.shape[:-1]
        x_b = x.reshape(*bsz, self.n_blocks, self.block_in)
        
        # FIX: Changed 'bio' to 'boi' to match (n_blocks, block_out, block_in)
        block_out = torch.einsum('...bi,boi->...bo', x_b, self.blocks)
        
        block_out = block_out.reshape(*bsz, self.n_blocks * self.block_out)
        mix_out = (x @ self.mix_down.t()) @ self.mix_up.t()
        return block_out + mix_out

    def param_count(self):
        return self.blocks.numel() + self.mix_down.numel() + self.mix_up.numel()

class LowRankLinear(nn.Module):
    def __init__(self, in_f, out_f, rank=128):
        super().__init__()
        self.A = nn.Parameter(torch.randn(out_f, rank) * 0.02)
        self.B = nn.Parameter(torch.randn(rank, in_f) * 0.02)
    def forward(self, x):
        return (x @ self.B.t()) @ self.A.t()
    def param_count(self):
        return self.A.numel() + self.B.numel()

# ============================================================
# Mini Transformer
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return F.rms_norm(x, (x.size(-1),)) * self.scale

class Attention(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, make_linear):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.q_size = num_heads * self.head_dim
        self.kv_size = num_kv_heads * self.head_dim
        # QKV always standard (not testing alternatives here)
        self.c_qkv = nn.Linear(dim, self.q_size + 2 * self.kv_size, bias=False)
        # Output projection: THIS is what we're testing
        self.proj = make_linear(dim, dim)

    def forward(self, x):
        B, S, D = x.shape
        qkv = self.c_qkv(x)
        q, k, v = qkv.split([self.q_size, self.kv_size, self.kv_size], dim=-1)
        q = q.reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        # Expand KV for GQA
        rep = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        # Standard attention
        scale = 1.0 / math.sqrt(self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * scale
        causal_mask = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
        attn = attn.masked_fill(causal_mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        y = (attn @ v).transpose(1, 2).reshape(B, S, D)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, dim, hidden, make_linear_fc, make_linear_proj):
        super().__init__()
        self.fc = make_linear_fc(dim, hidden)
        self.proj = make_linear_proj(hidden, dim)
    def forward(self, x):
        return self.proj(F.relu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, hidden,
                 make_attn_proj, make_mlp_fc, make_mlp_proj):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = Attention(dim, num_heads, num_kv_heads, make_attn_proj)
        self.norm2 = RMSNorm(dim)
        self.mlp = MLP(dim, hidden, make_mlp_fc, make_mlp_proj)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab, dim, n_layers, num_heads, num_kv_heads, hidden,
                 make_attn_proj, make_mlp_fc, make_mlp_proj):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([
            Block(dim, num_heads, num_kv_heads, hidden,
                  make_attn_proj, make_mlp_fc, make_mlp_proj)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.tok_emb.weight = self.head.weight  # tie

    def forward(self, idx):
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

# ============================================================
# Training + Eval
# ============================================================

def train_and_eval(model, label, n_steps=N_STEPS):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    
    n_params = sum(p.numel() for p in model.parameters())
    # Count only the variant-specific params (attn proj + MLP fc + MLP proj per layer)
    variant_params = 0
    for block in model.blocks:
        variant_params += block.attn.proj.param_count()
        variant_params += block.mlp.fc.param_count()
        variant_params += block.mlp.proj.param_count()
    
    results = []
    
    # Warmup
    model.train()
    for _ in range(5):
        idx = torch.randint(0, len(train_data), (BATCH,), device=device)
        batch = train_data[idx]
        logits = model(batch[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1))
        loss.backward()
        opt.zero_grad()
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    t_start = time.perf_counter()
    
    for step in range(1, n_steps + 1):
        model.train()
        idx = torch.randint(0, len(train_data), (BATCH,), device=device)
        batch = train_data[idx]
        logits = model(batch[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad()
        
        if step % EVAL_EVERY == 0 or step == n_steps:
            model.eval()
            with torch.no_grad():
                val_logits = model(val_data[:, :-1])
                val_loss = F.cross_entropy(val_logits.reshape(-1, VOCAB), val_data[:, 1:].reshape(-1)).item()
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - t_start
            ms_per_step = elapsed / step * 1000
            
            results.append((step, val_loss, ms_per_step))
    
    final_loss = results[-1][1]
    final_ms = results[-1][2]
    ternary_mb = variant_params * 1.6 / 8 / 1e6 * N_LAYERS  # extrapolate to 12L
    
    return {
        'label': label,
        'val_loss': final_loss,
        'ms_step': final_ms,
        'total_params': n_params,
        'variant_params': variant_params,
        'ternary_12L_MB': ternary_mb,
        'history': results,
    }

# ============================================================
# Configurations to test
# ============================================================

def std(in_f, out_f):
    return StandardLinear(in_f, out_f)

configs = {
    # Baseline: all standard linear
    'All Standard': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: StandardLinear(i, o),
        'mlp_proj':  lambda i, o: StandardLinear(i, o),
    },
    
    # Grouped MLP only (attn proj stays standard — needs cross-head mixing)
    'MLP Grouped g=2': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=2),
    },
    'MLP Grouped g=4': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=4),
    },
    
    # Grouped everywhere
    'All Grouped g=2': {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=2),
    },
    'All Grouped g=4': {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=4),
    },

    # BlockDiag+Mix on MLP (recovers cross-group info)
    'MLP BD4+mix32': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: BlockDiagPlusMix(i, o, n_blocks=4, mix_rank=32),
        'mlp_proj':  lambda i, o: BlockDiagPlusMix(i, o, n_blocks=4, mix_rank=32),
    },

    # Low-rank MLP
    'MLP LowRank r=128': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: LowRankLinear(i, o, rank=128),
        'mlp_proj':  lambda i, o: LowRankLinear(i, o, rank=128),
    },
    'MLP LowRank r=256': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: LowRankLinear(i, o, rank=256),
        'mlp_proj':  lambda i, o: LowRankLinear(i, o, rank=256),
    },

    # Attn proj variants (MLP stays standard)
    'Attn Grouped g=2': {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_fc':    lambda i, o: StandardLinear(i, o),
        'mlp_proj':  lambda i, o: StandardLinear(i, o),
    },
    'Attn LowRank r=128': {
        'attn_proj': lambda i, o: LowRankLinear(i, o, rank=128),
        'mlp_fc':    lambda i, o: StandardLinear(i, o),
        'mlp_proj':  lambda i, o: StandardLinear(i, o),
    },

    # Hybrid: grouped MLP + grouped attn at different group counts
    'Attn g=2, MLP g=4': {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=4),
    },
}

# ============================================================
# Run all configs
# ============================================================

print(f'\nMini-Transformer: {N_LAYERS}L {DIM}d {NUM_HEADS}h {NUM_KV_HEADS}kv MLP={MLP_MULT}x')
print(f'Training: {N_STEPS} steps, batch={BATCH}, seq={SEQ_LEN}, vocab={VOCAB}')
print()
print(f'{"Config":<28} {"ValLoss":>8} {"ms/step":>8} {"TotParams":>10} {"VarParams":>10} {"Tern12L":>8} {"Notes":>20}')
print('-' * 105)

all_results = []
baseline_loss = None

for name, cfg in configs.items():
    model = MiniGPT(
        VOCAB, DIM, N_LAYERS, NUM_HEADS, NUM_KV_HEADS, HIDDEN,
        make_attn_proj=cfg['attn_proj'],
        make_mlp_fc=cfg['mlp_fc'],
        make_mlp_proj=cfg['mlp_proj'],
    )
    result = train_and_eval(model, name)
    all_results.append(result)
    
    if baseline_loss is None:
        baseline_loss = result['val_loss']
    delta = result['val_loss'] - baseline_loss
    sign = '+' if delta >= 0 else ''
    notes = f'{sign}{delta:.4f} vs base'
    
    print(f'{name:<28} {result["val_loss"]:>8.4f} {result["ms_step"]:>7.1f}ms {result["total_params"]:>10,} {result["variant_params"]:>10,} {result["ternary_12L_MB"]:>7.2f}M {notes:>20}')

# ============================================================
# Pareto analysis
# ============================================================
print('\n' + '=' * 105)
print('TRAINING CURVES (val_loss at checkpoints)')
print('=' * 105)
for r in all_results:
    curve = ' | '.join([f'step {s}: {l:.4f}' for s, l, _ in r['history']])
    print(f'{r["label"]:<28} {curve}')

Device: cuda

Mini-Transformer: 3L 768d 8h 4kv MLP=2x
Training: 400 steps, batch=32, seq=128, vocab=512

Config                        ValLoss  ms/step  TotParams  VarParams  Tern12L                Notes
---------------------------------------------------------------------------------------------------------
All Standard                   4.5070    10.8ms 12,784,896  8,847,360    5.31M      +0.0000 vs base
MLP Grouped g=2                4.4956     9.5ms  9,245,952  5,308,416    3.19M      -0.0114 vs base
MLP Grouped g=4                4.4608     8.3ms  7,476,480  3,538,944    2.12M      -0.0461 vs base
All Grouped g=2                4.5211     9.3ms  8,361,216  4,423,680    2.65M      +0.0142 vs base
All Grouped g=4                4.4709     8.1ms  6,149,376  2,211,840    1.33M      -0.0361 vs base
MLP BD4+mix32                  4.4395     9.4ms  7,918,848  3,981,312    2.39M      -0.0675 vs base
MLP LowRank r=128              4.4308     8.0ms  7,476,480  3,538,944    2.12M      -0.076